# BAYYINAH — Legal Corpus Ingestion

### قبل التشغيل:
- **Colab**: `Runtime → Change runtime type → T4 GPU`
- **Kaggle**: `Settings → Accelerator → GPU T4 x2`

ثم شغّل كل الـ cells بالترتيب.

In [ ]:
# -- Cell 2: Credentials --------------------------------------------------
# Set credentials before running. Do NOT hardcode them here.
# In Colab:  left panel -> key icon (Secrets) -> add QDRANT_URL etc.
# In Kaggle: Add-ons -> Secrets.
import os
QDRANT_URL        = os.environ.get('QDRANT_URL',        'https://YOUR-CLUSTER.cloud.qdrant.io')
QDRANT_API_KEY    = os.environ.get('QDRANT_API_KEY',    'YOUR-JWT-API-KEY')
QDRANT_COLLECTION = os.environ.get('QDRANT_COLLECTION', 'egypt_legal_rag')
BATCH_SIZE        = 64

In [ ]:
# -- Cell 2: Credentials --------------------------------------------------
# Set credentials before running. Do NOT hardcode them here.
# In Colab:  left panel -> key icon (Secrets) -> add QDRANT_URL etc.
# In Kaggle: Add-ons -> Secrets.
import os
QDRANT_URL        = os.environ.get('QDRANT_URL',        'https://YOUR-CLUSTER.cloud.qdrant.io')
QDRANT_API_KEY    = os.environ.get('QDRANT_API_KEY',    'YOUR-JWT-API-KEY')
QDRANT_COLLECTION = os.environ.get('QDRANT_COLLECTION', 'egypt_legal_rag')
BATCH_SIZE        = 64

In [ ]:
# ── Cell 3: Verify GPU ────────────────────────────────────────────────────────
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — enable GPU in runtime settings first!")

In [ ]:
# ── Cell 4: Download & Chunk ──────────────────────────────────────────────────
import re
from datasets import load_dataset

LAW_TYPE_MAP = {
    "احوال_شخصية": "family",  "الأحوال_الشخصية": "family",
    "الاحوال_الشخصية": "family", "ميراث": "family", "وصية": "family",
    "عمل": "labor", "العمل": "labor", "تأمينات": "labor",
    "مدني": "civil", "المدني": "civil", "إيجار": "civil", "تجاري": "civil",
    "عقوبات": "criminal", "العقوبات": "criminal", "إجراءات_جنائية": "criminal",
    "مرافعات": "procedural", "المرافعات": "procedural",
    "دستور": "constitutional",
}

ARTICLE_RE = re.compile(r'(?:ال)?مادة\s*(?:رقم\s*)?\(?([\d\u0660-\u0669]+)\)?', re.UNICODE)

def detect_law_type(name):
    n = name.replace(" ", "_")
    for k, v in LAW_TYPE_MAP.items():
        if k in n: return v
    return "other"

def detect_num_year(text):
    m = re.search(r'رقم\s+"?(\d+)"?\s+لسنة\s+(\d{4})', text[:600])
    return (m.group(1), m.group(2)) if m else (None, None)

def normalize_digits(s):
    return s.translate(str.maketrans('\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669', '0123456789'))

def chunk_law(law_name, categories, text):
    law_type    = detect_law_type(law_name)
    law_num, yr = detect_num_year(text)
    display     = law_name.replace("_", " ").strip()
    slug        = re.sub(r'\W+', '_', law_name)[:28]
    cat         = categories[0] if categories else ""
    base = dict(doc_id=slug, law_name=display, law_number=law_num or "",
                law_year=yr or "", law_type=law_type, category=cat, cross_references=[])
    matches = list(ARTICLE_RE.finditer(text))
    if not matches:
        t = text[:3000].strip()
        return [{**base, "chunk_id": f"{slug}_full", "article_number": "", "text": t, "context_text": t}]
    chunks = []
    for i, m in enumerate(matches):
        art_num = normalize_digits(m.group(1))
        end     = matches[i+1].start() if i+1 < len(matches) else len(text)
        t       = text[m.start():end].strip()[:2000]
        if len(t) < 20: continue
        chunks.append({**base, "chunk_id": f"{slug}_art{art_num}", "article_number": art_num, "text": t, "context_text": t})
    return chunks

print("Downloading egypt-legal-corpus...")
dataset = load_dataset("dataflare/egypt-legal-corpus", split="train")
print(f"Laws: {len(dataset)}")

all_chunks, seen = [], set()
for row in dataset:
    for ch in chunk_law(row["law_name"], row.get("categories", []), row["text"]):
        cid = ch["chunk_id"]
        if cid in seen: cid = f"{cid}_{len(seen)}"
        seen.add(cid); ch["chunk_id"] = cid
        all_chunks.append(ch)

print(f"Articles extracted: {len(all_chunks)}")
s = all_chunks[100]
print(f"Sample: {s['law_name']} | art {s['article_number']} | {s['text'][:80]}")

In [ ]:
# ── Cell 5: Load BGE-M3 ───────────────────────────────────────────────────────
from FlagEmbedding import BGEM3FlagModel
print(f"Loading BAAI/bge-m3 on {device} ...")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(device=="cuda"), device=device)
test  = model.encode(["test"], batch_size=1, max_length=32, return_dense=True)
print(f"Ready. dim={test['dense_vecs'].shape[1]}")

In [ ]:
# ── Cell 6: Connect Qdrant ────────────────────────────────────────────────────
from qdrant_client import QdrantClient
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=30)
info   = client.get_collection(QDRANT_COLLECTION)
print(f"Connected. Points now: {info.points_count}")

In [ ]:
# ── Cell 7: Ingest ────────────────────────────────────────────────────────────
import time
from qdrant_client.http import models as qmodels

total, t0 = len(all_chunks), time.time()
print(f"Ingesting {total} articles...")

for start in range(0, total, BATCH_SIZE):
    batch   = all_chunks[start:start+BATCH_SIZE]
    out     = model.encode([ch["text"] for ch in batch], batch_size=BATCH_SIZE, max_length=512, return_dense=True)
    vectors = out["dense_vecs"].tolist()
    client.upsert(
        collection_name=QDRANT_COLLECTION,
        points=[
            qmodels.PointStruct(id=abs(hash(ch["chunk_id"]))%(2**63), vector={"dense": vectors[i]}, payload=ch)
            for i, ch in enumerate(batch)
        ]
    )
    done = start+len(batch)
    rate = done/(time.time()-t0)
    eta  = (total-done)/rate/60 if rate else 0
    print(f"  {done}/{total} [{int(done/total*100)}%]  {rate:.0f} art/s  ETA {eta:.1f} min")

print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Cell 8: Verify ────────────────────────────────────────────────────────────
info = client.get_collection(QDRANT_COLLECTION)
print(f"Vectors stored: {info.points_count}")

for q in ["هل يجوز فصل العامل بدون سبب؟", "ما هي حقوق الزوجة في الطلاق؟"]:
    qvec = model.encode([q], return_dense=True)["dense_vecs"][0].tolist()
    res  = client.query_points(collection_name=QDRANT_COLLECTION, query=qvec, using="dense", limit=3, with_payload=True).points
    print(f"\nQ: {q}")
    for r in res:
        p = r.payload
        print(f"  {r.score:.4f} | {p.get('law_name')} | art {p.get('article_number')} | {p.get('text','')[:80]}")